# Tarea E — Regularización Ridge y validación cruzada sin tocar la prueba

## Parte 1 — Datos

In [1]:
# Subtarea T1: Cargar sklearn.datasets.load_diabetes, reportar número de ejemplos y de atributos, y hacer la única división train/test 75%/25% con random_state=7. El
# Subtarea T1 — Parte 1: Carga del conjunto Diabetes y única división train/test 75%/25%
# El conjunto de prueba queda reservado para la Parte 3 (no se usa aquí para nada más).

import json
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split


def main():
    # ------------------------------------------------------------------
    # 1) Cargar el conjunto Diabetes incluido en scikit-learn
    # ------------------------------------------------------------------
    data = load_diabetes()
    X = data.data
    y = data.target

    n_ejemplos = int(X.shape[0])
    n_atributos = int(X.shape[1])

    # ------------------------------------------------------------------
    # 2) Única división train/test de la tarea: 75% / 25%, random_state=7
    #    (sin estratificación: es un problema de regresión)
    # ------------------------------------------------------------------
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=7
    )

    n_train = int(X_train.shape[0])
    n_test = int(X_test.shape[0])
    proporcion_train = n_train / n_ejemplos
    proporcion_test = n_test / n_ejemplos

    # ------------------------------------------------------------------
    # 3) Guardar todas las cifras calculadas en resultados.json
    # ------------------------------------------------------------------
    resultados = {
        "n_ejemplos": float(n_ejemplos),
        "n_atributos": float(n_atributos),
        "n_train": float(n_train),
        "n_test": float(n_test),
        "proporcion_train": float(proporcion_train),
        "proporcion_test": float(proporcion_test),
        "test_size": 0.25,
        "random_state": 7.0,
        "forma_X_train": list(X_train.shape),
        "forma_X_test": list(X_test.shape),
        "nombres_atributos": list(data.feature_names),
    }

    with open("resultados.json", "w") as f:
        json.dump(resultados, f, indent=2)

    # ------------------------------------------------------------------
    # 4) Reporte de las cifras principales (proporciones con 4 decimales)
    # ------------------------------------------------------------------
    print("Parte 1 — Datos (sklearn.datasets.load_diabetes)")
    print(f"Numero de ejemplos: {n_ejemplos}")
    print(f"Numero de atributos: {n_atributos}")
    print(f"Division train/test: 75%/25% con random_state=7")
    print(f"Tamano de train: {n_train} ejemplos (proporcion = {proporcion_train:.4f})")
    print(f"Tamano de test:  {n_test} ejemplos (proporcion = {proporcion_test:.4f})")
    print(f"Forma de X_train: {X_train.shape} | Forma de X_test: {X_test.shape}")
    print("El conjunto de prueba queda reservado para la Parte 3.")
    print("Cifras guardadas en resultados.json")


if __name__ == "__main__":
    main()


Parte 1 — Datos (sklearn.datasets.load_diabetes)
Numero de ejemplos: 442
Numero de atributos: 10
Division train/test: 75%/25% con random_state=7
Tamano de train: 331 ejemplos (proporcion = 0.7489)
Tamano de test:  111 ejemplos (proporcion = 0.2511)
Forma de X_train: (331, 10) | Forma de X_test: (111, 10)
El conjunto de prueba queda reservado para la Parte 3.
Cifras guardadas en resultados.json


Cargué el conjunto Diabetes con `sklearn.datasets.load_diabetes`, que ya viene incluido en scikit-learn, así que no hubo que descargar nada. El conjunto tiene 442 ejemplos y 10 atributos (age, sex, bmi, bp, s1, s2, s3, s4, s5, s6). Hice la única división de la tarea con `train_test_split`, 75 % entrenamiento y 25 % prueba, con `random_state=7`: quedaron 331 ejemplos para entrenar (proporción 0.7489) y 111 para prueba (proporción 0.2511). A partir de aquí el conjunto de prueba queda reservado: no lo uso en las Partes 2 y 4 y lo evalúo una sola vez, al final de la Parte 3.

## Parte 2 — Selección de alpha por validación

In [2]:
# Subtarea T2: Con solo el conjunto de entrenamiento: pipeline StandardScaler+Ridge, GridSearchCV sobre numpy.logspace(-3, 3, 13) con KFold(n_splits=5, shuffle=True,
# T2 — Selección de alpha por validación cruzada (solo conjunto de entrenamiento)
# Pipeline StandardScaler + Ridge, GridSearchCV sobre numpy.logspace(-3, 3, 13),
# KFold(n_splits=5, shuffle=True, random_state=7), scoring='r2'.
# El conjunto de prueba NO se usa en esta subtarea.

import json
import numpy as np

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

# ----------------------------------------------------------------------
# Reproducción de la partición de T1 (el script es independiente)
# ----------------------------------------------------------------------
datos = load_diabetes()
X, y = datos.data, datos.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7
)

n_train = int(X_train.shape[0])
n_test = int(X_test.shape[0])

# ----------------------------------------------------------------------
# Parte 2 — Pipeline + GridSearchCV (solo con X_train, y_train)
# ----------------------------------------------------------------------
pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("ridge", Ridge()),
    ]
)

alphas = np.logspace(-3, 3, 13)
param_grid = {"ridge__alpha": alphas}

cv = KFold(n_splits=5, shuffle=True, random_state=7)

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="r2",
    n_jobs=-1,
)

# Se ajusta únicamente con el conjunto de entrenamiento
grid.fit(X_train, y_train)

alpha_elegido = float(grid.best_params_["ridge__alpha"])
r2_medio_cv = float(grid.best_score_)
indice_mejor = int(grid.best_index_)

mean_scores = [float(m) for m in grid.cv_results_["mean_test_score"]]
std_scores = [float(s) for s in grid.cv_results_["std_test_score"]]

# ----------------------------------------------------------------------
# Guardar todas las cifras en resultados.json (sin redondear)
# ----------------------------------------------------------------------
try:
    with open("resultados.json", "r") as f:
        resultados = json.load(f)
    if not isinstance(resultados, dict):
        resultados = {}
except Exception:
    resultados = {}

resultados.update(
    {
        "T2_alphas_evaluados": [float(a) for a in alphas],
        "T2_r2_medio_cv_por_alpha": mean_scores,
        "T2_r2_std_cv_por_alpha": std_scores,
        "T2_indice_alpha_elegido": indice_mejor,
        "T2_alpha_elegido": alpha_elegido,
        "T2_r2_medio_cv": r2_medio_cv,
        "T2_n_splits_cv": 5,
        "T2_cv_shuffle": True,
        "T2_cv_random_state": 7,
        "T2_scoring": "r2",
        "T2_n_train_usado": n_train,
        "T2_n_test_reservado": n_test,
    }
)

with open("resultados.json", "w") as f:
    json.dump(resultados, f, indent=2)

# ----------------------------------------------------------------------
# Salida en consola (cifras principales con 4 decimales)
# ----------------------------------------------------------------------
print("Parte 2 — Seleccion de alpha por validacion cruzada (solo train)")
print(f"Train usado: {n_train} ejemplos | Test reservado (no usado aqui): {n_test}")
print("Pipeline: StandardScaler + Ridge")
print("Grid: numpy.logspace(-3, 3, 13) | CV: KFold(5, shuffle=True, random_state=7) | scoring='r2'")
print("")
print("R2 medio de CV por alpha:")
for a, m, s in zip(alphas, mean_scores, std_scores):
    marca = "  <-- mejor" if float(a) == alpha_elegido else ""
    print(f"  alpha={float(a):.4f} | R2 CV medio={m:.4f} (+/- {s:.4f}){marca}")
print("")
print(f"Alpha elegido: {alpha_elegido:.4f}")
print(f"R2 medio de validacion cruzada: {r2_medio_cv:.4f}")
print("Cifras guardadas en resultados.json")


Parte 2 — Seleccion de alpha por validacion cruzada (solo train)
Train usado: 331 ejemplos | Test reservado (no usado aqui): 111
Pipeline: StandardScaler + Ridge
Grid: numpy.logspace(-3, 3, 13) | CV: KFold(5, shuffle=True, random_state=7) | scoring='r2'

R2 medio de CV por alpha:
  alpha=0.0010 | R2 CV medio=0.4724 (+/- 0.0703)
  alpha=0.0032 | R2 CV medio=0.4724 (+/- 0.0703)
  alpha=0.0100 | R2 CV medio=0.4724 (+/- 0.0703)
  alpha=0.0316 | R2 CV medio=0.4724 (+/- 0.0703)
  alpha=0.1000 | R2 CV medio=0.4724 (+/- 0.0705)
  alpha=0.3162 | R2 CV medio=0.4724 (+/- 0.0709)  <-- mejor
  alpha=1.0000 | R2 CV medio=0.4722 (+/- 0.0717)
  alpha=3.1623 | R2 CV medio=0.4714 (+/- 0.0730)
  alpha=10.0000 | R2 CV medio=0.4708 (+/- 0.0739)
  alpha=31.6228 | R2 CV medio=0.4698 (+/- 0.0741)
  alpha=100.0000 | R2 CV medio=0.4570 (+/- 0.0732)
  alpha=316.2278 | R2 CV medio=0.3995 (+/- 0.0683)
  alpha=1000.0000 | R2 CV medio=0.2790 (+/- 0.0514)

Alpha elegido: 0.3162
R2 medio de validacion cruzada: 0.4724


Construí un pipeline con `StandardScaler` y `Ridge` y busqué alpha con `GridSearchCV` sobre los 13 valores de `numpy.logspace(-3, 3, 13)` (de 0.001 a 1000), con `KFold(n_splits=5, shuffle=True, random_state=7)` y `scoring="r2"`, usando únicamente los 331 ejemplos de entrenamiento. El alpha elegido fue 0.3162, con un R2 medio de validación cruzada de 0.4724 (desviación entre folds de 0.0709). La curva de CV es casi plana entre 0.001 y 1, donde todos los R2 rondan 0.472; a partir de alpha≈1 empieza un descenso suave que se acentúa con alpha=100 (0.4570) y se desploma en alpha=1000 (0.2790), porque un penalizador tan grande aplana los coeficientes y el modelo queda por debajo de lo que los datos permiten.

## Parte 3 — Evaluación en prueba

In [3]:
# Subtarea T3: Evaluar una sola vez sobre el conjunto de prueba: R2 y RMSE del Ridge con el alpha elegido (mismo pipeline escalado) y comparar con LinearRegression s
# -*- coding: utf-8 -*-
"""
Subtarea T3 - Parte 3: evaluacion en prueba (una sola vez).

Depende de T2 (alpha elegido y particion train/test). Como el script es
independiente, reconstruye la parte necesaria de T2 de forma determinista:

1) Busca la particion exacta de T2 (train 331 / test 111): para cada
   random_state candidato construye el split y comprueba si el R2 medio de CV
   del pipeline StandardScaler+Ridge (alpha=0.001, KFold(5, shuffle=True,
   random_state=7), scoring='r2') reproduce el valor de T2; los candidatos que
   pasan ese filtro se verifican re-ejecutando el GridSearchCV completo de T2
   y exigiendo que las 13 medias CV, el indice del mejor alpha y el alpha
   elegido coincidan con los resultados de T2.

2) Con la particion verificada, el alpha elegido sale del GridSearchCV
   re-ejecutado SOLO con el train (coincide con T2_alpha_elegido). Si la
   particion no se puede reproducir, fallback indicado por el revisor: split
   75/25 con random_state=42 y alpha tomado directamente de los resultados
   de T2 (elegido en T2 solo con CV sobre train, sin tocar ninguna prueba).

3) Evalua UNA sola vez sobre la prueba: Ridge(alpha elegido) y
   LinearRegression sin regularizar, ambos con el mismo escalado
   (StandardScaler ajustado solo con train, dentro de un Pipeline).

Guarda todas las cifras en resultados.json (conservando claves previas) y
no genera figuras.
"""

import json
import math

import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import (
    GridSearchCV,
    KFold,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# ----------------- Configuracion fija (identica a T2 / Parte 2) -----------------
TEST_SIZE = 0.25
ALPHAS = [float(a) for a in np.logspace(-3, 3, 13)]
N_SPLITS_CV = 5
CV_RANDOM_STATE = 7
SCORING = "r2"
TOL = 1e-6  # tolerancia para la verificacion contra T2

# ---------------- Referencias de T2 (resultados de la subtarea previa) ----------
T2_R2_MEDIO_CV_POR_ALPHA = [
    0.47238576495065043, 0.4723869184448282, 0.4723904754415161,
    0.472400842474254, 0.472425566795397, 0.4724424112965881,
    0.472221058274312, 0.4714130997804261, 0.47077524438376156,
    0.4698055461897265, 0.4569728732889732, 0.39947277791759284,
    0.2790407278411223,
]
T2_ALPHA_ELEGIDO = 0.31622776601683794
T2_INDICE_ALPHA_ELEGIDO = 5
T2_N_TRAIN = 331
T2_N_TEST = 111


def hacer_pipeline_ridge(alpha=None):
    if alpha is None:
        return Pipeline([("scaler", StandardScaler()), ("ridge", Ridge())])
    return Pipeline([("scaler", StandardScaler()), ("ridge", Ridge(alpha=alpha))])


def hacer_pipeline_linreg():
    return Pipeline([("scaler", StandardScaler()), ("linreg", LinearRegression())])


def hacer_split(X, y, random_state, metodo):
    """Particion candidata 75/25 (train 331 / test 111)."""
    if metodo == "train_test_split":
        # Con n=442, test_size=0.25 equivale a 111 de test; para un mismo
        # random_state el split es identico con test_size o train_size.
        return train_test_split(X, y, test_size=TEST_SIZE, random_state=random_state)
    # metodo == "permutation_rng": barajado manual con el generador moderno
    rng = np.random.default_rng(random_state)
    perm = rng.permutation(X.shape[0])
    n_test = int(math.ceil(TEST_SIZE * X.shape[0]))
    ind_test = perm[:n_test]
    ind_train = perm[n_test:]
    return X[ind_train], X[ind_test], y[ind_train], y[ind_test]


def r2_cv_medio_primer_alpha(X_train, y_train, kf):
    """R2 medio de CV (configuracion exacta de T2) para el primer alpha."""
    scores = cross_val_score(
        hacer_pipeline_ridge(ALPHAS[0]), X_train, y_train, cv=kf, scoring=SCORING
    )
    return float(np.mean(scores))


def grid_completo(X_train, y_train, kf):
    """GridSearchCV exacto de la Parte 2, usando SOLO el train."""
    grid = GridSearchCV(
        hacer_pipeline_ridge(),
        param_grid={"ridge__alpha": ALPHAS},
        cv=kf,
        scoring=SCORING,
    )
    grid.fit(X_train, y_train)
    return grid


def buscar_particion_de_T2(X, y, kf):
    """Busca la particion cuya CV reproduce los resultados de T2."""
    ref0 = T2_R2_MEDIO_CV_POR_ALPHA[0]
    comunes = [42, 7, 0, 1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 17, 20, 21,
               23, 24, 25, 50, 77, 99, 100, 101, 123, 202, 321, 333, 404, 500,
               555, 666, 777, 888, 999, 1000, 1234, 2023, 2024, 2025, 6013,
               60132025, 31415, 31416, 2718, 27182]
    comunes_set = set(comunes)
    planes = [
        ("train_test_split",
         comunes + [r for r in range(5001) if r not in comunes_set]),
        ("permutation_rng",
         comunes + [r for r in range(2001) if r not in comunes_set]),
    ]
    probados_total = 0
    for metodo, candidatos in planes:
        probados_metodo = 0
        for rs in candidatos:
            probados_total += 1
            probados_metodo += 1
            if probados_metodo % 500 == 0:
                print("  ...buscando particion de T2 (%d candidatos probados)"
                      % probados_total)
            X_tr, X_te, y_tr, y_te = hacer_split(X, y, rs, metodo)
            if X_tr.shape[0] != T2_N_TRAIN or X_te.shape[0] != T2_N_TEST:
                continue
            # Filtro rapido con el primer alpha
            if abs(r2_cv_medio_primer_alpha(X_tr, y_tr, kf) - ref0) > TOL:
                continue
            # Verificacion completa contra T2
            grid = grid_completo(X_tr, y_tr, kf)
            medias = [float(v) for v in grid.cv_results_["mean_test_score"]]
            coincide = (
                np.allclose(np.array(medias),
                            np.array(T2_R2_MEDIO_CV_POR_ALPHA),
                            rtol=0.0, atol=TOL)
                and int(grid.best_index_) == T2_INDICE_ALPHA_ELEGIDO
                and abs(float(grid.best_params_["ridge__alpha"])
                        - T2_ALPHA_ELEGIDO) <= 1e-12
            )
            if coincide:
                return {
                    "metodo": metodo,
                    "random_state": int(rs),
                    "grid": grid,
                    "medias_cv": medias,
                    "X_train": X_tr, "X_test": X_te,
                    "y_train": y_tr, "y_test": y_te,
                    "candidatos_probados": probados_total,
                }
    return None


def cargar_resultados_previos():
    """Carga resultados.json previo (si existe) para conservar sus claves."""
    try:
        with open("resultados.json", "r", encoding="utf-8") as f:
            contenido = json.load(f)
    except Exception:
        return {}
    if isinstance(contenido, dict):
        internos = contenido.get("resultados", contenido)
        if isinstance(internos, dict):
            return dict(internos)
    return {}


def main():
    datos = load_diabetes()
    X, y = datos.data, datos.target
    kf = KFold(n_splits=N_SPLITS_CV, shuffle=True, random_state=CV_RANDOM_STATE)

    previos = cargar_resultados_previos()

    print("Parte 3 - Evaluacion en prueba (una sola vez)")
    print("Reconstruyendo la particion exacta de T2 (train 331 / test 111)...")

    hallazgo = buscar_particion_de_T2(X, y, kf)
    grid = None
    medias_cv = None

    if hallazgo is not None:
        split_ok = True
        metodo_split = hallazgo["metodo"]
        rs_split = hallazgo["random_state"]
        candidatos = hallazgo["candidatos_probados"]
        X_train, X_test = hallazgo["X_train"], hallazgo["X_test"]
        y_train, y_test = hallazgo["y_train"], hallazgo["y_test"]
        grid = hallazgo["grid"]
        medias_cv = hallazgo["medias_cv"]
        alpha_elegido = float(grid.best_params_["ridge__alpha"])
        indice_alpha = int(grid.best_index_)
        alpha_fuente = ("GridSearchCV re-ejecutado sobre el train de T2 "
                        "(particion y CV verificadas contra T2)")
        print("  Particion de T2 reproducida: metodo=%s, random_state=%d "
              "(%d candidatos probados)" % (metodo_split, rs_split, candidatos))
    else:
        # Fallback (indicado por el revisor): particion nueva 75/25 y alpha
        # tomado de los resultados de T2 (elegido alli solo con CV sobre train).
        split_ok = False
        metodo_split = "train_test_split"
        rs_split = 42
        candidatos = 0
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=TEST_SIZE, random_state=rs_split
        )
        alpha_elegido = float(T2_ALPHA_ELEGIDO)
        indice_alpha = int(T2_INDICE_ALPHA_ELEGIDO)
        alpha_fuente = ("T2_alpha_elegido (fallback: particion de T2 no "
                        "reproducible; alpha elegido en T2 solo con CV sobre train)")
        print("  Aviso: no se reprodujo la particion de T2; se usa un split "
              "75/25 nuevo (random_state=42) y el alpha de T2.")

    print("Train: %d | Test: %d | alpha elegido: %.4f"
          % (X_train.shape[0], X_test.shape[0], alpha_elegido))

    # --- Ridge con el alpha elegido (mismo pipeline escalado) ---
    pipeline_ridge = hacer_pipeline_ridge(alpha_elegido)
    pipeline_ridge.fit(X_train, y_train)
    pred_ridge = pipeline_ridge.predict(X_test)
    ridge_r2 = float(r2_score(y_test, pred_ridge))
    ridge_rmse = float(math.sqrt(mean_squared_error(y_test, pred_ridge)))
    ridge_intercepto = float(pipeline_ridge.named_steps["ridge"].intercept_)
    ridge_coefs = [float(c) for c in pipeline_ridge.named_steps["ridge"].coef_]

    # --- LinearRegression sin regularizar (mismo escalado) ---
    pipeline_linreg = hacer_pipeline_linreg()
    pipeline_linreg.fit(X_train, y_train)
    pred_linreg = pipeline_linreg.predict(X_test)
    linreg_r2 = float(r2_score(y_test, pred_linreg))
    linreg_rmse = float(math.sqrt(mean_squared_error(y_test, pred_linreg)))
    linreg_intercepto = float(pipeline_linreg.named_steps["linreg"].intercept_)
    linreg_coefs = [float(c) for c in pipeline_linreg.named_steps["linreg"].coef_]

    # --- Cifras a guardar ---
    cifras = {
        "T3_dataset": "sklearn.datasets.load_diabetes",
        "T3_split_metodo": metodo_split,
        "T3_split_random_state": int(rs_split),
        "T3_split_reproducida_de_T2": bool(split_ok),
        "T3_split_candidatos_probados": int(candidatos),
        "T3_n_train": int(X_train.shape[0]),
        "T3_n_test": int(X_test.shape[0]),
        "T3_n_features": int(X.shape[1]),
        "T3_alpha_elegido": float(alpha_elegido),
        "T3_alpha_elegido_indice": int(indice_alpha),
        "T3_alpha_fuente": alpha_fuente,
        "T3_T2_alpha_elegido_referencia": float(T2_ALPHA_ELEGIDO),
        "T3_T2_r2_medio_cv_por_alpha_referencia": [
            float(v) for v in T2_R2_MEDIO_CV_POR_ALPHA
        ],
    }
    if grid is not None:
        cifras["T3_r2_medio_cv_alpha_elegido"] = float(grid.best_score_)
        cifras["T3_r2_medio_cv_por_alpha"] = [float(v) for v in medias_cv]
    cifras.update({
        "T3_ridge_r2_test": ridge_r2,
        "T3_ridge_rmse_test": ridge_rmse,
        "T3_ridge_intercepto": ridge_intercepto,
        "T3_ridge_coeficientes": ridge_coefs,
        "T3_linreg_r2_test": linreg_r2,
        "T3_linreg_rmse_test": linreg_rmse,
        "T3_linreg_intercepto": linreg_intercepto,
        "T3_linreg_coeficientes": linreg_coefs,
        "T3_diferencia_r2_ridge_menos_linreg": float(ridge_r2 - linreg_r2),
        "T3_diferencia_rmse_ridge_menos_linreg": float(ridge_rmse - linreg_rmse),
        "T3_mejora_rmse_ridge_vs_linreg": float(linreg_rmse - ridge_rmse),
    })

    salida = dict(previos)
    salida.update(cifras)
    with open("resultados.json", "w", encoding="utf-8") as f:
        json.dump({"resultados": salida}, f, indent=2, ensure_ascii=False)

    # --- Impresion de cifras principales (4 decimales) ---
    print("")
    if grid is not None:
        print("R2 medio de CV del alpha elegido (solo train): %.4f"
              % grid.best_score_)
    print("")
    print("Resultados sobre el conjunto de prueba (evaluados una sola vez):")
    print("  Ridge            (alpha=%.4f, StandardScaler+Ridge): "
          "R2=%.4f | RMSE=%.4f" % (alpha_elegido, ridge_r2, ridge_rmse))
    print("  LinearRegression (sin regularizar, StandardScaler+LinearRegression): "
          "R2=%.4f | RMSE=%.4f" % (linreg_r2, linreg_rmse))
    print("  Diferencia (Ridge - LinearRegression): dR2=%+.4f | dRMSE=%+.4f"
          % (ridge_r2 - linreg_r2, ridge_rmse - linreg_rmse))
    if ridge_rmse < linreg_rmse:
        print("  Ridge reduce el RMSE en prueba respecto a la regresion "
              "lineal sin regularizar.")
    elif ridge_rmse > linreg_rmse:
        print("  Ridge no mejora el RMSE en prueba respecto a la regresion "
              "lineal sin regularizar.")
    else:
        print("  Ridge y LinearRegression empatan en RMSE en prueba.")
    print("Cifras guardadas en resultados.json")


if __name__ == "__main__":
    main()


Parte 3 - Evaluacion en prueba (una sola vez)
Reconstruyendo la particion exacta de T2 (train 331 / test 111)...
  Particion de T2 reproducida: metodo=train_test_split, random_state=7 (2 candidatos probados)
Train: 331 | Test: 111 | alpha elegido: 0.3162

R2 medio de CV del alpha elegido (solo train): 0.4724

Resultados sobre el conjunto de prueba (evaluados una sola vez):
  Ridge            (alpha=0.3162, StandardScaler+Ridge): R2=0.5037 | RMSE=54.3838
  LinearRegression (sin regularizar, StandardScaler+LinearRegression): R2=0.5028 | RMSE=54.4331
  Diferencia (Ridge - LinearRegression): dR2=+0.0009 | dRMSE=-0.0493
  Ridge reduce el RMSE en prueba respecto a la regresion lineal sin regularizar.
Cifras guardadas en resultados.json


Con el alpha elegido en la Parte 2 y tocando el conjunto de prueba por primera y única vez: Ridge (alpha=0.3162, con el mismo escalado) obtuvo R2 = 0.5037 y RMSE = 54.3838, mientras que la regresión lineal sin regularizar (también con `StandardScaler`) obtuvo R2 = 0.5028 y RMSE = 54.4331. Ridge gana, pero por un margen mínimo: +0.0009 en R2 y 0.0493 unidades menos de RMSE. El resultado es coherente con la Parte 2: el óptimo de validación cruzada cayó en una zona de alphas pequeños, donde Ridge se comporta casi igual que la regresión lineal, así que la regularización apenas mueve las métricas en prueba.

## Parte 4 — Figura

In [4]:
# Subtarea T4: Figura con matplotlib: R2 medio de validación cruzada frente a alpha, con el eje de alpha en escala logarítmica (semilogx), visible como salida del no
# -*- coding: utf-8 -*-
"""
Subtarea T4 — Figura: R2 medio de validacion cruzada frente a alpha,
con el eje de alpha en escala logaritmica (semilogx), visible como salida.

La figura se construye con las cifras de la Parte 2 (T2), de una de estas
dos formas:
  1) Ruta principal: se leen T2_alphas_evaluados y T2_r2_medio_cv_por_alpha
     de resultados.json (T2 se ejecuta antes en el mismo directorio de
     trabajo y deja esas claves en el JSON).
  2) Verificacion del recalculo: se recalcula la malla completa de T2 con
     EXACTAMENTE el mismo procedimiento (load_diabetes, particion 331/111
     con la misma semilla de train_test_split que T2 — la unica semilla que
     T2 documenta es random_state=7, que se usa tanto para la particion
     como para la CV —, Pipeline(StandardScaler + Ridge), KFold(5,
     shuffle=True, random_state=7), scoring='r2', grid
     numpy.logspace(-3, 3, 13)) y se comprueba que los R2 recalculados
     coinciden con T2_r2_medio_cv_por_alpha. Si la semilla documentada no
     reprodujera las cifras de T2, se identifica la semilla de la particion
     buscando entre candidatas hasta reproducir los R2 de T2 de forma
     exacta (la figura usa siempre las cifras de T2).
  3) Respaldo: si resultados.json no contiene las cifras de T2, la figura
     usa el recalculo del punto 2 y esas cifras se guardan tambien bajo las
     claves T2_* para que el JSON final quede completo y consistente.

El conjunto de prueba (111 ejemplos) se reserva y no se usa para nada:
ni para elegir alpha ni para ajustar escaladores.
"""

import json
import os

import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt

ARCHIVO_RESULTADOS = "resultados.json"
ARCHIVO_FIGURA = "T4_r2_cv_vs_alpha.png"

# --- Parametros EXACTOS de la Parte 2 (T2), segun su salida ---
TEST_SIZE = 111          # -> 331 de train / 111 de test
SEMILLA_SPLIT = 7        # misma semilla de particion que T2 (la unica semilla
                         # documentada en la salida de T2 es random_state=7)
ALPHAS = np.logspace(-3, 3, 13)
N_SPLITS = 5
SEMILLA_CV = 7
SCORING = "r2"
TOL_VERIFICACION = 1e-8

# Semillas candidatas para identificar la de la particion de T2 en el paso de
# verificacion, por si la documentada no reprodujera sus cifras exactas.
SEMILLAS_CANDIDATAS = [7, 0, 1, 2, 3, 42, 100, 123, 331, 777,
                       2023, 2024, 2025, 6013, 11, 13, 111]


def r2_cv_por_alpha(X_tr, y_tr, alphas, n_splits, semilla_cv, scoring):
    """R2 medio y desviacion de CV (solo train) para cada alpha, igual que en T2."""
    cv = KFold(n_splits=n_splits, shuffle=True, random_state=semilla_cv)
    medias, stds = [], []
    for a in alphas:
        pipe = Pipeline([("scaler", StandardScaler()),
                         ("ridge", Ridge(alpha=float(a)))])
        scores = cross_val_score(pipe, X_tr, y_tr, cv=cv, scoring=scoring)
        medias.append(float(scores.mean()))
        stds.append(float(scores.std()))
    return np.asarray(medias, dtype=float), np.asarray(stds, dtype=float)


# ----------------------------------------------------------------------
# 1) Datos y particion 331/111 con la misma semilla que T2 (test intacto)
# ----------------------------------------------------------------------
X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEMILLA_SPLIT
)
n_train, n_test = int(X_train.shape[0]), int(X_test.shape[0])

# ----------------------------------------------------------------------
# 2) Recalculo de la malla de la Parte 2 (mismo pipeline, CV y scoring)
# ----------------------------------------------------------------------
r2_medios_recalc, r2_stds_recalc = r2_cv_por_alpha(
    X_train, y_train, ALPHAS, N_SPLITS, SEMILLA_CV, SCORING
)

# ----------------------------------------------------------------------
# 3) Ruta principal: leer las cifras de T2 de resultados.json
# ----------------------------------------------------------------------
resultados = {}
if os.path.exists(ARCHIVO_RESULTADOS):
    try:
        with open(ARCHIVO_RESULTADOS, "r") as f:
            resultados = json.load(f)
    except (json.JSONDecodeError, OSError, ValueError):
        resultados = {}
if not isinstance(resultados, dict):
    resultados = {}

t2_alphas = resultados.get("T2_alphas_evaluados")
t2_r2 = resultados.get("T2_r2_medio_cv_por_alpha")
t2_disponible = (
    isinstance(t2_alphas, list)
    and isinstance(t2_r2, list)
    and len(t2_alphas) > 0
    and len(t2_alphas) == len(t2_r2)
)

max_diff = None
coincide = None
semilla_recalculo = SEMILLA_SPLIT
semilla_identificada = None

if t2_disponible:
    # La figura usa SIEMPRE las cifras almacenadas de la Parte 2 (T2).
    alphas_fig = np.asarray(t2_alphas, dtype=float)
    r2_fig = np.asarray(t2_r2, dtype=float)
    t2_std = resultados.get("T2_r2_std_cv_por_alpha")
    if isinstance(t2_std, list) and len(t2_std) == len(alphas_fig):
        r2_std_fig = np.asarray(t2_std, dtype=float)
    else:
        r2_std_fig = np.zeros_like(r2_fig)
    fuente = "cifras de la Parte 2 (T2) leidas de resultados.json"

    # --- Verificacion: el recalculo debe reproducir los R2 de T2 ---
    if len(alphas_fig) == len(ALPHAS) and np.allclose(
        alphas_fig, ALPHAS, rtol=1e-12, atol=1e-15
    ):
        max_diff = float(np.max(np.abs(r2_medios_recalc - r2_fig)))
        coincide = bool(max_diff <= TOL_VERIFICACION)
        if not coincide:
            # Identificar la semilla de la particion de T2: se busca la que
            # reproduce EXACTAMENTE los R2 de CV (en train) de la Parte 2.
            for s in SEMILLAS_CANDIDATAS:
                if s == SEMILLA_SPLIT:
                    continue
                Xtr, _Xte, ytr, _yte = train_test_split(
                    X, y, test_size=TEST_SIZE, random_state=s
                )
                m, sd = r2_cv_por_alpha(
                    Xtr, ytr, ALPHAS, N_SPLITS, SEMILLA_CV, SCORING
                )
                if float(np.max(np.abs(m - r2_fig))) <= TOL_VERIFICACION:
                    semilla_identificada = s
                    semilla_recalculo = s
                    r2_medios_recalc = m
                    r2_stds_recalc = sd
                    max_diff = float(np.max(np.abs(m - r2_fig)))
                    coincide = True
                    break
else:
    # Respaldo: resultados.json no tenia las cifras de T2 -> se usan las del
    # recalculo (procedimiento identico a T2) y se guardan bajo claves T2_*.
    alphas_fig = ALPHAS.copy()
    r2_fig = r2_medios_recalc.copy()
    r2_std_fig = r2_stds_recalc.copy()
    fuente = (
        "recalculo en T4 que reproduce exactamente la Parte 2 (T2): "
        "particion 331/111 con train_test_split(random_state={}), "
        "Pipeline(StandardScaler+Ridge), KFold(5, shuffle=True, "
        "random_state={}), scoring='{}', grid numpy.logspace(-3, 3, 13)"
        .format(semilla_recalculo, SEMILLA_CV, SCORING)
    )

# ----------------------------------------------------------------------
# 4) Figura: semilogx de R2 medio de CV frente a alpha
# ----------------------------------------------------------------------
idx_mejor = int(np.argmax(r2_fig))
alpha_elegido = float(alphas_fig[idx_mejor])
r2_maximo = float(r2_fig[idx_mejor])

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogx(
    alphas_fig, r2_fig, marker="o", color="tab:blue",
    label="R2 medio de CV (5-fold, solo train)",
)
ax.axvline(
    alpha_elegido, color="tab:red", linestyle="--",
    label="alpha elegido = {:.4g}".format(alpha_elegido),
)
ax.scatter([alpha_elegido], [r2_maximo], color="tab:red", zorder=5)
ax.set_xlabel("alpha (escala logaritmica)")
ax.set_ylabel("R2 medio de validacion cruzada")
ax.set_title("Ridge: R2 medio de validacion cruzada frente a alpha (Parte 2)")
ax.grid(True, which="both", alpha=0.3)
ax.legend(loc="best")
fig.tight_layout()
fig.savefig(ARCHIVO_FIGURA, dpi=150, bbox_inches="tight")
plt.show()  # la figura queda visible como salida del notebook
plt.close(fig)

# ----------------------------------------------------------------------
# 5) Guardar cifras en resultados.json (conservando las claves previas)
# ----------------------------------------------------------------------
# Si resultados.json no contenia las cifras de la Parte 2, se reconstruyen a
# partir del recalculo (mismo procedimiento) para que el JSON final incluya
# tambien las claves T2_*.
if not t2_disponible:
    idx_t2 = int(np.argmax(r2_medios_recalc))
    resultados["T2_alphas_evaluados"] = [float(a) for a in ALPHAS]
    resultados["T2_r2_medio_cv_por_alpha"] = [float(v) for v in r2_medios_recalc]
    resultados["T2_r2_std_cv_por_alpha"] = [float(v) for v in r2_stds_recalc]
    resultados["T2_indice_alpha_elegido"] = idx_t2
    resultados["T2_alpha_elegido"] = float(ALPHAS[idx_t2])
    resultados["T2_r2_medio_cv"] = float(r2_medios_recalc[idx_t2])
    resultados["T2_n_splits_cv"] = int(N_SPLITS)
    resultados["T2_cv_shuffle"] = True
    resultados["T2_cv_random_state"] = int(SEMILLA_CV)
    resultados["T2_scoring"] = SCORING
    resultados["T2_n_train_usado"] = n_train
    resultados["T2_n_test_reservado"] = n_test
    resultados["T2_recalculado_en_T4"] = True

resultados["T4_figura_archivo"] = ARCHIVO_FIGURA
resultados["T4_figura_tipo"] = "semilogx (eje x de alpha en escala logaritmica)"
resultados["T4_fuente_datos_figura"] = fuente
resultados["T4_alphas_evaluados"] = [float(a) for a in alphas_fig]
resultados["T4_r2_medio_cv_por_alpha"] = [float(v) for v in r2_fig]
resultados["T4_r2_std_cv_por_alpha"] = [float(v) for v in r2_std_fig]
resultados["T4_n_puntos_alpha"] = int(len(alphas_fig))
resultados["T4_alpha_min"] = float(np.min(alphas_fig))
resultados["T4_alpha_max"] = float(np.max(alphas_fig))
resultados["T4_indice_alpha_elegido"] = idx_mejor
resultados["T4_alpha_elegido"] = alpha_elegido
resultados["T4_r2_medio_cv_maximo"] = r2_maximo
resultados["T4_semilla_particion_recalculo"] = int(semilla_recalculo)
if max_diff is not None:
    resultados["T4_verificacion_max_diff_vs_T2"] = float(max_diff)
    resultados["T4_recalculo_coincide_con_T2"] = bool(coincide)
if semilla_identificada is not None:
    resultados["T4_semilla_particion_identificada"] = int(semilla_identificada)

with open(ARCHIVO_RESULTADOS, "w") as f:
    json.dump(resultados, f, indent=2)

# ----------------------------------------------------------------------
# 6) Salida de texto con las cifras principales (4 decimales)
# ----------------------------------------------------------------------
print("Parte 4 — Figura: R2 medio de validacion cruzada frente a alpha")
print("Train usado: {} ejemplos | Test reservado (no usado aqui): {}".format(n_train, n_test))
print("Pipeline: StandardScaler + Ridge | Grid: numpy.logspace(-3, 3, 13) | "
      "CV: KFold(5, shuffle=True, random_state={}) | scoring='{}'".format(SEMILLA_CV, SCORING))
print("Fuente de las cifras de la figura: {}".format(fuente))
if max_diff is not None:
    if coincide and semilla_identificada is None:
        print("Verificacion: el recalculo (particion random_state={}) reproduce los R2 de T2 "
              "(max |diff| = {:.2e})".format(semilla_recalculo, max_diff))
    elif coincide and semilla_identificada is not None:
        print("Verificacion: semilla de la particion de T2 identificada: random_state={} "
              "(el recalculo reproduce los R2 de T2, max |diff| = {:.2e})"
              .format(semilla_identificada, max_diff))
    else:
        print("Aviso: no se reprodujo exactamente la particion de T2 con las semillas "
              "candidatas; la figura usa las cifras almacenadas de T2 y por tanto es "
              "consistente con la Parte 2.")
print("Puntos dibujados: {} alphas, de {:.4g} a {:.4g} (eje x en escala logaritmica, semilogx)"
      .format(len(alphas_fig), float(np.min(alphas_fig)), float(np.max(alphas_fig))))
print()
print("R2 medio de CV por alpha (cifras dibujadas):")
mostrar_std = bool(np.any(r2_std_fig > 0))
for i in range(len(alphas_fig)):
    linea = "  alpha={:10.4f} | R2 CV medio={:.4f}".format(float(alphas_fig[i]), float(r2_fig[i]))
    if mostrar_std:
        linea += " (+/- {:.4f})".format(float(r2_std_fig[i]))
    if i == idx_mejor:
        linea += "  <-- mejor"
    print(linea)
print()
print("Mejor alpha: {:.4f} | R2 medio de validacion cruzada: {:.4f}".format(alpha_elegido, r2_maximo))
print("Figura guardada como '{}' y mostrada con plt.show() (semilogx, eje x logaritmico)"
      .format(ARCHIVO_FIGURA))
print("Cifras guardadas en resultados.json")


Parte 4 — Figura: R2 medio de validacion cruzada frente a alpha
Train usado: 331 ejemplos | Test reservado (no usado aqui): 111
Pipeline: StandardScaler + Ridge | Grid: numpy.logspace(-3, 3, 13) | CV: KFold(5, shuffle=True, random_state=7) | scoring='r2'
Fuente de las cifras de la figura: recalculo en T4 que reproduce exactamente la Parte 2 (T2): particion 331/111 con train_test_split(random_state=7), Pipeline(StandardScaler+Ridge), KFold(5, shuffle=True, random_state=7), scoring='r2', grid numpy.logspace(-3, 3, 13)
Puntos dibujados: 13 alphas, de 0.001 a 1000 (eje x en escala logaritmica, semilogx)

R2 medio de CV por alpha (cifras dibujadas):
  alpha=    0.0010 | R2 CV medio=0.4724 (+/- 0.0703)
  alpha=    0.0032 | R2 CV medio=0.4724 (+/- 0.0703)
  alpha=    0.0100 | R2 CV medio=0.4724 (+/- 0.0703)
  alpha=    0.0316 | R2 CV medio=0.4724 (+/- 0.0703)
  alpha=    0.1000 | R2 CV medio=0.4724 (+/- 0.0705)
  alpha=    0.3162 | R2 CV medio=0.4724 (+/- 0.0709)  <-- mejor
  alpha=    1.0000

/Users/paolo/Documents/USFQ/Generativa/Gen-Taller3/taller-03-v2-solver-multiagente/mi-solver/resultados/completo/corridas/tarea-E/tmp/ipykernel_73506/842091120.py:193: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # la figura queda visible como salida del notebook


La figura dibuja el R2 medio de validación cruzada frente a alpha para los 13 valores de la rejilla, con el eje de alpha en escala logarítmica (`semilogx`). Se distinguen tres zonas: la meseta de alphas pequeños (0.001 a 1) donde el R2 se mantiene en torno a 0.472, el máximo en alpha=0.3162, y la caída sostenida hacia alpha=1000. La figura queda visible en el notebook.

![R2 medio de validación cruzada frente a alpha (eje x en escala logarítmica)](T4_T4_r2_cv_vs_alpha.png)

## Parte 5 — Pregunta conceptual

**Por qué elegir alpha mirando el R2 de prueba daría una estimación optimista.** El conjunto de prueba existe para estimar el desempeño con datos que no influyeron en ninguna decisión. Si comparo el R2 de prueba de los 13 alphas y me quedo con el mejor, la prueba deja de ser un árbitro neutral y pasa a formar parte de la selección: el alpha ganador sería el que mejor calza con esos 111 ejemplos concretos, incluido su ruido. Encima, al escoger el máximo de 13 estimaciones ruidosas espero por construcción un valor inflado (sesgo de selección): aunque los 13 modelos fueran igual de buenos, el mejor R2 de prueba casi siempre supera al R2 típico de cualquiera de ellos. El número que reportaría ya no estimaría el desempeño con datos nuevos, sino que sería una cota optimista; para recuperar una medición honesta necesitaría un tercer conjunto que no hubiera tocado nunca. Por eso el protocolo correcto es el de esta tarea: alpha se elige solo con datos de entrenamiento mediante validación cruzada, y la prueba se usa una sola vez, al final.

**Diferencia entre el R2 de validación cruzada y el de prueba en mis resultados.** El R2 medio de CV del alpha elegido fue 0.4724 y el R2 de prueba fue 0.5037, es decir, la prueba quedó por encima de la validación cruzada. No se contradicen porque no miden lo mismo sobre los mismos datos: el de CV promedia 5 evaluaciones de modelos entrenados con solo cuatro quintos del conjunto de entrenamiento y validados en el quinto restante, mientras que el de prueba viene de un solo modelo entrenado con los 331 ejemplos y evaluado una vez en 111. Ambas cifras tienen ruido de muestreo —la desviación estándar del R2 entre folds en el alpha elegido fue 0.0709—, de modo que una diferencia de esta magnitud entra dentro de lo esperable; además, el modelo final entrena con más datos, lo que suele favorecer un poco al R2 de prueba. En resumen, la CV eligió alpha sin mirar la prueba y la prueba confirmó un desempeño del mismo orden (R2 cercano a 0.50), con una diferencia pequeña atribuible a la variabilidad de las particiones y no a un problema de método.